# Week 3 — Context Engineering II

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-03-context-engineering-ii-content.html`.

**You will practice:**
1. Sliding-window truncation vs. rolling summarization on a synthetic conversation.
2. Generating embeddings and computing cosine similarity.
3. A minimal semantic search function (the seed of next week's RAG system).
4. A light ADK / LangChain preview of the summarization call.
5. Two open exercises.


In [ ]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv numpy pydantic

In [3]:
import os
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"
EMBED_MODEL = "gemini-embedding-001"

## 1. Sliding window vs. rolling summarization

In [1]:
history = [
    {"role": "user", "content": "Hi, I'm planning a trip to Peru."},
    {"role": "model", "content": "Great! When are you thinking of traveling, and for how long?"},
    {"role": "user", "content": "Sometime in August, for about 2 weeks."},
    {"role": "model", "content": "August is dry season in the Andes, good for Machu Picchu. Any budget in mind?"},
    {"role": "user", "content": "Around $2000 total, excluding flights."},
    {"role": "model", "content": "That's workable for hostels and local transport. Interested in the Amazon too, or just the highlands?"},
    {"role": "user", "content": "Just the highlands. I don't do well with humidity."},
    {"role": "model", "content": "Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is."},
    {"role": "user", "content": "Also I'm vegetarian, will that be an issue?"},
    {"role": "model", "content": "Not at all, Peruvian highland cuisine has plenty of vegetarian options."},
]

def sliding_window(history, max_turns=4):
    return history[-max_turns:]

print("--- Sliding window (last 4) ---")
for m in sliding_window(history, max_turns=4):
    print(m["role"], ":", m["content"])

--- Sliding window (last 4) ---
user : Just the highlands. I don't do well with humidity.
model : Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is.
user : Also I'm vegetarian, will that be an issue?
model : Not at all, Peruvian highland cuisine has plenty of vegetarian options.


In [4]:
def summarize_history(history):
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history)
    prompt = (
        "Summarize the key facts, decisions, and constraints from this conversation "
        "in 3-5 bullet points. Be specific.\n\n" + transcript
    )
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

def compact_context(history, keep_recent=4):
    if len(history) <= keep_recent:
        return history
    older, recent = history[:-keep_recent], history[-keep_recent:]
    summary = summarize_history(older)
    return [{"role": "system", "content": f"Conversation summary so far:\n{summary}"}] + recent

compacted = compact_context(history, keep_recent=4)
print("--- Rolling summarization + last 4 ---")
for m in compacted:
    print(m["role"], ":", m["content"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


--- Rolling summarization + last 4 ---
system : Conversation summary so far:
Here is a summary of the key facts, decisions, and constraints from the conversation:

* **Destination & Timing:** The user is planning a roughly two-week trip to Peru in August, which aligns with the dry season in the Andes.
* **Budget Constraint:** The total budget is capped at approximately $2,000 (excluding flights), which is suitable for hostels and local transportation.
* **Pending Decision on Scope:** The itinerary will include the Andean highlands (Machu Picchu), with a pending decision on whether to also include the Amazon rainforest.
user : Just the highlands. I don't do well with humidity.
model : Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is.
user : Also I'm vegetarian, will that be an issue?
model : Not at all, Peruvian highland cuisine has plenty of vegetarian options.


## 2. Embeddings and cosine similarity

In [5]:
def embed(text):
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return result.embeddings[0].values

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

v_brake = embed("The bike's front brake feels loose.")
v_caliper = embed("The caliper on the front wheel isn't gripping properly.")
v_weather = embed("It might rain this weekend.")

print("brake vs caliper (related): ", round(cosine_similarity(v_brake, v_caliper), 3))
print("brake vs weather (unrelated):", round(cosine_similarity(v_brake, v_weather), 3))

brake vs caliper (related):  0.804
brake vs weather (unrelated): 0.543


## 3. A minimal semantic search function

In [6]:
candidates = [
    "To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.",
    "Flat tires are usually caused by punctures or worn-out inner tubes.",
    "Our rental bikes come in small, medium, and large frame sizes.",
    "Helmets are provided free of charge with every rental.",
    "The gear shifter may need cable tension adjustment if shifting feels sluggish.",
    "Rentals can be extended by messaging support at least 2 hours before the due time.",
]
candidate_vectors = [embed(c) for c in candidates]

def semantic_search(query, top_k=3):
    q_vec = embed(query)
    scored = [(cosine_similarity(q_vec, v), text) for v, text in zip(candidate_vectors, candidates)]
    scored.sort(reverse=True)
    return scored[:top_k]

for score, text in semantic_search("my brake feels wobbly and doesn't stop well"):
    print(f"{score:.3f}  {text}")

0.767  To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.
0.617  The gear shifter may need cable tension adjustment if shifting feels sluggish.
0.579  Flat tires are usually caused by punctures or worn-out inner tubes.


## 4. Light preview: the summarization call via ADK and LangChain

Same idea as previous weeks — reproducing one call (this time, `summarize_history`) through different tools.

In [7]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

summarizer_agent = Agent(
    model=MODEL,
    name="summarizer_agent",
    instruction="Summarize the given conversation in 3-5 specific bullet points.",
)

async def ask_adk_agent(agent, prompt, app_name="week3_app", user_id="student"):
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = await runner.session_service.create_session(app_name=app_name, user_id=user_id)
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history[:-4])
print("ADK summary:\n", await ask_adk_agent(summarizer_agent, transcript))

Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


ADK summary:
 Here is a summary of the conversation:

* The user is planning a two-week trip to Peru during August.
* The user established a travel budget of around $2,000, excluding international flights.
* The assistant confirmed the budget is suitable for hostels and local transit, and noted that August is the dry season in the Andes (ideal for Machu Picchu).
* The assistant asked whether the user wants to include the Amazon in the itinerary or focus solely on the highlands.


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model=MODEL)
lc_summary = llm.invoke(
    "Summarize the key facts, decisions, and constraints from this conversation in 3-5 bullet points:\n\n"
    + transcript
)
print("LangChain summary:\n", lc_summary.content)

# LangChain also wraps the embedding model the same way:
lc_embeddings = GoogleGenerativeAIEmbeddings(model=f"models/{EMBED_MODEL}")
lc_vector = lc_embeddings.embed_query("The bike's front brake feels loose.")
print("\nLangChain embedding dims:", len(lc_vector))

LangChain summary:
 [{'type': 'text', 'text': 'Here is a summary of the key facts, decisions, and constraints from the conversation:\n\n* **Destination & Timing:** The user is planning an approximately two-week trip to Peru in August, which falls during the Andean dry season.\n* **Budget Constraint:** The total budget is around $2,000 (excluding international flights), which is suitable for budget accommodations (hostels) and local transportation.\n* **Pending Decision:** The scope of the itinerary is yet to be decided, specifically whether to explore only the Andean highlands (e.g., Machu Picchu) or also include the Amazon.', 'extras': {'signature': 'EvUOCvIOARFNMg/mQfeekicVFgdQKhUqzz7npiIwUPTrBODsU2f4dKndmTHGp6+wPKDufZZ3LG3g6zkDc1GX9+sdzxHNrjGF80CK8A8drCvSDlE43tsL46zHFLDnMIhj1jDUq1ZXWtdBkkYYSuJPuBgBZGYpLXbFF8aIyOt+8L93Fu5hlytr1j8WMu0foXoTL1ELVyTjkLQbT/uFItyfUU9NkZfeQivjeGR+odSi3+g417/EDseeimCkjA4SO70Rgh2BnWWaDt+vbeHb6hkPIthI065VoiECzi/LaqGmu88LlXPsk8hIXtUkfPm2qs3pKkj327T99UD6t9XcISat

## 5. Exercises

In [9]:
# Exercise 1 — Ranked semantic search with scores

def semantic_search_v2(query, top_k=3, threshold=0.0):
    q_vec = embed(query)
    scored = [(cosine_similarity(q_vec, v), text) for v, text in zip(candidate_vectors, candidates)]
    scored.sort(reverse=True)
    scored = [(s, t) for s, t in scored if s >= threshold]
    return scored[:top_k]

print("--- query with a good match, threshold=0.5 ---")
for score, text in semantic_search_v2("my brake feels wobbly and doesn't stop well", top_k=3, threshold=0.5):
    print(f"{score:.3f}  {text}")

print("\n--- query with no good match, threshold=0.5 ---")
results = semantic_search_v2("what's the weather like on Mars", top_k=3, threshold=0.5)
if not results:
    print("(no results above threshold)")
for score, text in results:
    print(f"{score:.3f}  {text}")

--- query with a good match, threshold=0.5 ---
0.767  To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.
0.617  The gear shifter may need cable tension adjustment if shifting feels sluggish.
0.579  Flat tires are usually caused by punctures or worn-out inner tubes.

--- query with no good match, threshold=0.5 ---
(no results above threshold)


In [10]:
# Exercise 2 — Structured memory extraction

class TripMemory(BaseModel):
    destination: str
    month: str
    budget_usd: float
    dietary_restrictions: list[str]
    regions_of_interest: list[str]

full_transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history)
resp = client.models.generate_content(
    model=MODEL,
    contents=f"Extract the trip details from this conversation:\n\n{full_transcript}",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=TripMemory,
    ),
)
trip: TripMemory = resp.parsed
print(trip)

destination='Peru' month='August' budget_usd=2000.0 dietary_restrictions=['vegetarian'] regions_of_interest=['highlands']


## Next week

Week 4 — **Retrieval-Augmented Generation (RAG)**: chunking strategies, ChromaDB and FAISS, and building a
complete, functional RAG system — the foundation of your Corte 1 project. See `week-04-rag-content.html`.